# Exp 41: Best vs fixed 0.5 thresholds

Card: [exp41_best_vs_fixed_05.md](../../experiments/exp41_best_vs_fixed_05.md)


In [ ]:
import os

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import precision_recall_curve

from core.config import CLASSES, ROOT
from core.cv import VIP, CVResult, Config

CONFIG = Config.from_toml(
    ROOT / "sections" / "model" / "configs" / "exp29_downsample_5fold" / "resnet18.toml"
)
FIXED = (0.5, 0.5, 0.5, 0.5, 0.5)  # per class, CLASSES order
SMOKE = os.environ.get("SMOKE") == "1"
SAVE_FIGURES = not SMOKE  # only reads saved results, smoke changes nothing else
FIGURE = (
    ROOT / "sections" / "model" / "figures" / "exp41_best_vs_fixed_05_pr_curves.png"
)

# dataviz reference palette, light mode
INK, BEST, FIXED_COLOR = "#52514e", "#eb6834", "#2a78d6"
plt.rcParams.update(
    {
        "figure.facecolor": "#fcfcfb",
        "axes.facecolor": "#fcfcfb",
        "savefig.facecolor": "#fcfcfb",
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.edgecolor": INK,
        "xtick.color": INK,
        "ytick.color": INK,
        "axes.labelcolor": INK,
        "axes.titlesize": "medium",
        "axes.titlecolor": INK,
        "axes.grid": True,
        "grid.color": "#e1e0d9",
    }
)

## F1 per class and Macro F1


In [ ]:
scores = CVResult.load(CONFIG).scores  # pooled OOF rows of folds 0-4
points = {
    "best-F1": scores.best_f1(),
    "all 0.5": scores.fixed_f1(FIXED),
}  # threshold, precision, recall, F1
table = pd.concat({rule: p[["threshold", "f1"]] for rule, p in points.items()}, axis=1)
table.loc["Macro F1"] = [
    float("nan") if col == "threshold" else table[(rule, col)].mean()
    for rule, col in table.columns
]
table.round(3)

## PR curves, VIP classes


In [ ]:
fig, axes = plt.subplots(1, len(VIP), figsize=(3.6 * len(VIP), 3.6), sharey=True)
for cls, ax in zip(VIP, axes):
    c = CLASSES.index(cls)
    precision, recall, _ = precision_recall_curve(
        scores.y_true[:, c], scores.probs[:, c]
    )
    ax.plot(recall, precision, color=INK, lw=2, drawstyle="steps-post")
    for (rule, p), color in zip(points.items(), (BEST, FIXED_COLOR)):
        row = p.loc[cls]
        ax.plot(
            row["recall"],
            row["precision"],
            "o",
            ms=8,
            color=color,
            mec="#fcfcfb",
            mew=2,
            label=f"{rule}: t {row['threshold']:.3f}, F1 {row['f1']:.3f}",
        )
    ax.legend(loc="lower left", fontsize=8, frameon=False, labelcolor=INK)
    ax.set(
        title=f"{cls} ({int(scores.y_true[:, c].sum())} positives)",
        xlabel="recall",
        xlim=(0, 1.02),
        ylim=(0, 1.05),
    )
axes[0].set_ylabel("precision")
fig.suptitle("exp29 ResNet-18, out-of-fold rows of folds 0–4", color=INK)
fig.tight_layout()
if SAVE_FIGURES:
    fig.savefig(FIGURE, dpi=150, bbox_inches="tight")
plt.show()